# DepthWizard V2 - Direct Metric Elevation Engine & Category MAE Optimization
### ISRO SIH26175: Monocular Satellite DSM Estimation & 3D Terrain Reconstruction

**Core Upgrades in this Run:**
1. **Direct Physical Metric Regression**: Model outputs physical metres directly without external calibrators or `45m` scaling.
2. **Category-Targeted Loss Formulation**:
   - **Tall Skyscrapers (>40m)**: Focal exponent weighting $w_{\text{tall}} = 1 + 2.0 \cdot (y / 20)^{1.5}$ (capped at 6.0x) to eliminate height ceilings.
   - **Canopy & Trees (2–12m)**: Asymmetric crown penalty ($1.35\times$) when $\hat{y} < y$ to capture LiDAR first-return foliage.
   - **Urban Buildings & Offices**: Multi-scale spatial gradient loss ($\lambda_{\text{grad}} = 0.8$) across 4 pyramid scales for sharp parapet walls.
   - **Ground / Bare Earth (<1.5m)**: Total variation flatness regularizer to eliminate shadow pit artifacts.
3. **Resolution Invariance Augmentation**: 50% random multi-scale spatial blur ($1.3\times - 3.0\times$ area downsampling) to prevent error collapse on 2m satellite sensors (Cartosat/Sentinel-2).
4. **Exact Preprocessing**: $518 \times 518$ ViT patch-14 alignment with standard ImageNet normalization.
5. **Depth Anything V2 Base Backbone**: Scaled parameter capacity for large urban scene reasoning.

In [ ]:
# 1. Environment & Setup
import os, glob, random, time, json, math
import h5py
import numpy as np
from scipy.stats import pearsonr
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"[Setup] Device Initialized: {device} ({gpu_name})")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = True

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)

In [ ]:
# 2. Locate Mounted GAMUS Datasets (50GB Collection)
search_paths = glob.glob('/kaggle/input/**/images/**/*_RGB.h5', recursive=True)
if not search_paths:
    search_paths = glob.glob('/kaggle/input/**/*_RGB.h5', recursive=True)

def find_agl_file(rgb_p):
    candidates = [
        rgb_p.replace('/images/', '/heights/').replace('_RGB.h5', '_AGL.h5'),
        rgb_p.replace('_RGB.h5', '_AGL.h5'),
        rgb_p.replace('/images/', '/heights/').replace('_RGB.h5', '_DSM.h5'),
        rgb_p.replace('_RGB.h5', '_DSM.h5')
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

valid_pairs = []
for p in search_paths:
    agl_p = find_agl_file(p)
    if agl_p:
        valid_pairs.append((p, agl_p))

print(f"[Data] Found {len(valid_pairs)} verified optical satellite + LiDAR elevation tile pairs!")
if valid_pairs:
    print(f"  Sample RGB: {valid_pairs[0][0]}")
    print(f"  Sample AGL: {valid_pairs[0][1]}")

In [ ]:
# 3. Blur Augmentation & Direct Metric Dataset Loader
def apply_blur_augmentation(x: torch.Tensor, p: float = 0.5) -> torch.Tensor:
    if random.random() >= p:
        return x
    B, C, H, W = x.shape
    factor = random.uniform(1.3, 3.0)
    h_low = max(64, int(H / factor))
    w_low = max(64, int(W / factor))
    low_res = F.interpolate(x, size=(h_low, w_low), mode='area')
    return F.interpolate(low_res, size=(H, W), mode='bilinear', align_corners=False)

class MassiveGAMUSDataset(Dataset):
    def __init__(self, pairs, in_size=518, crop_size=512, crops_per_tile=16, is_train=True):
        self.pairs = pairs
        self.in_size = in_size
        self.crop_size = crop_size
        self.crops_per_tile = crops_per_tile
        self.is_train = is_train
        
    def __len__(self):
        return max(len(self.pairs) * self.crops_per_tile, 1)
        
    def __getitem__(self, idx):
        if not self.pairs:
            return torch.zeros((3, self.in_size, self.in_size)), torch.zeros((self.in_size, self.in_size))
            
        pair_idx = (idx // self.crops_per_tile) % len(self.pairs)
        rgb_p, agl_p = self.pairs[pair_idx]
        
        try:
            with h5py.File(rgb_p, 'r') as fr, h5py.File(agl_p, 'r') as fa:
                rgb = np.array(fr['image'])
                agl = np.nan_to_num(np.array(fa['image']).astype(np.float32), 0.0)
        except Exception:
            rgb = np.zeros((self.crop_size, self.crop_size, 3), dtype=np.uint8)
            agl = np.zeros((self.crop_size, self.crop_size), dtype=np.float32)
            
        H, W = rgb.shape[:2]
        if H > self.crop_size and W > self.crop_size:
            top = random.randint(0, H - self.crop_size)
            left = random.randint(0, W - self.crop_size)
            crop_rgb = rgb[top:top+self.crop_size, left:left+self.crop_size].copy()
            crop_agl = agl[top:top+self.crop_size, left:left+self.crop_size].copy()
        else:
            crop_rgb = rgb.copy()
            crop_agl = agl.copy()
            
        if self.is_train:
            if random.random() > 0.5:
                crop_rgb = np.fliplr(crop_rgb).copy()
                crop_agl = np.fliplr(crop_agl).copy()
            if random.random() > 0.5:
                crop_rgb = np.flipud(crop_rgb).copy()
                crop_agl = np.flipud(crop_agl).copy()
            k = random.randint(0, 3)
            if k > 0:
                crop_rgb = np.rot90(crop_rgb, k).copy()
                crop_agl = np.rot90(crop_agl, k).copy()
                
        rgb_t = torch.from_numpy(crop_rgb).permute(2, 0, 1).float() / 255.0
        agl_t = torch.from_numpy(crop_agl).float().clamp(min=0.0, max=300.0)
        
        # Resize to 518x518 (patch-14 token multiple: 37x37 patches)
        if (rgb_t.shape[1], rgb_t.shape[2]) != (self.in_size, self.in_size):
            rgb_t = F.interpolate(rgb_t.unsqueeze(0), size=(self.in_size, self.in_size), mode='bilinear', align_corners=False).squeeze(0)
            agl_t = F.interpolate(agl_t.unsqueeze(0).unsqueeze(0), size=(self.in_size, self.in_size), mode='bilinear', align_corners=False).squeeze(0).squeeze(0)
            
        return rgb_t, agl_t

random.shuffle(valid_pairs)
n_train = max(int(len(valid_pairs) * 0.88), 1)
train_ds = MassiveGAMUSDataset(valid_pairs[:n_train], in_size=518, crop_size=512, crops_per_tile=16, is_train=True)
val_ds = MassiveGAMUSDataset(valid_pairs[n_train:], in_size=518, crop_size=512, crops_per_tile=2, is_train=False)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2)
print(f"[Data] Dataset Ready: {len(train_ds)} train crops | {len(val_ds)} val crops across {len(valid_pairs)} tiles!")

In [ ]:
# 4. Category-Targeted Metric Charbonnier Loss
class MetricCharbonnierLoss(nn.Module):
    def __init__(self, eps: float = 1e-3, gamma_tall: float = 2.0, tall_cap: float = 6.0, grad_weight: float = 0.8):
        super().__init__()
        self.eps_sq = eps ** 2
        self.gamma_tall = gamma_tall
        self.tall_cap = tall_cap
        self.grad_weight = grad_weight

    def multi_scale_grad_loss(self, p: torch.Tensor, y: torch.Tensor, valid: torch.Tensor, scales: int = 4) -> torch.Tensor:
        tot = torch.tensor(0.0, device=p.device)
        curr_p, curr_y, curr_v = p[:, None], y[:, None], valid[:, None]
        for s in range(scales):
            if s > 0:
                curr_p = F.avg_pool2d(curr_p, 2)
                curr_y = F.avg_pool2d(curr_y * curr_v.float(), 2)
                curr_v = F.avg_pool2d(curr_v.float(), 2) > 0.99
            d = torch.where(curr_v, curr_p - curr_y, torch.zeros_like(curr_p))
            gx = (d[:, :, :, 1:] - d[:, :, :, :-1]).abs() * (curr_v[:, :, :, 1:] & curr_v[:, :, :, :-1])
            gy = (d[:, :, 1:, :] - d[:, :, :-1, :]).abs() * (curr_v[:, :, 1:, :] & curr_v[:, :, :-1, :])
            valid_sum = curr_v.sum().clamp(min=1.0)
            tot = tot + (gx.sum() + gy.sum()) / valid_sum
        return tot / scales

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        valid = torch.isfinite(target) & (target >= 0.0)
        if not torch.any(valid):
            return torch.tensor(0.0, device=pred.device, requires_grad=True)

        y_val = torch.nan_to_num(target, 0.0)
        diff = pred - y_val

        # 1. Focal Tall Structure Weighting (Skyscrapers > 20m)
        w_tall = 1.0 + (self.gamma_tall * torch.pow(y_val / 20.0, 1.5)).clamp(0.0, self.tall_cap)

        # 2. Canopy Crown Asymmetry (2m - 15m)
        is_canopy = (y_val >= 2.0) & (y_val <= 15.0)
        w_canopy = torch.where(is_canopy & (diff < 0), 1.35, 1.0)
        w = w_tall * w_canopy

        charb = torch.sqrt(diff ** 2 + self.eps_sq)
        charb_loss = (charb * w * valid).sum() / (w * valid).sum().clamp(min=1.0)

        # 3. Bare Earth Flatness (terrain < 1.5m)
        is_ground = (y_val < 1.5) & valid
        if torch.any(is_ground):
            p_ground = torch.where(is_ground, pred, torch.zeros_like(pred))
            gx_g = (p_ground[:, :, 1:] - p_ground[:, :, :-1]).abs()[:, :, :-1]
            gy_g = (p_ground[:, 1:, :] - p_ground[:, :-1, :]).abs()[:, :-1, :]
            tv_loss = (gx_g.mean() + gy_g.mean()) * 0.05
        else:
            tv_loss = 0.0

        # 4. Multi-scale Parapet Edge Loss
        g_loss = self.multi_scale_grad_loss(pred, y_val, valid)
        return charb_loss + self.grad_weight * g_loss + tv_loss

In [ ]:
# 5. Load Backbone (Depth Anything V2 Base or Small)
from transformers import AutoModelForDepthEstimation

# Check for locally mounted Base weights first, else pull from HF
local_base_candidates = [
    '/kaggle/input/depth-anything-v2-base',
    '/kaggle/input/asterlu/depth-anything-v2-base'
]
model_id = 'depth-anything/Depth-Anything-V2-Base-hf'
for c in local_base_candidates:
    if os.path.exists(c):
        model_id = c
        break

print(f"[Model] Loading Backbone: {model_id}...")
try:
    model = AutoModelForDepthEstimation.from_pretrained(model_id).to(device)
    print(f"[Model] Successfully loaded Depth Anything V2 Base ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)!")
except Exception as e:
    print(f"[Model] Fallback to Small backbone: {e}")
    model_id = 'depth-anything/Depth-Anything-V2-Small-hf'
    model = AutoModelForDepthEstimation.from_pretrained(model_id).to(device)
    print(f"[Model] Successfully loaded Depth Anything V2 Small ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)!")

criterion = MetricCharbonnierLoss(gamma_tall=2.0, grad_weight=0.8)
epochs = 12
optimizer = torch.optim.AdamW(model.parameters(), lr=1.8e-5, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda' if torch.cuda.is_available() else 'cpu', enabled=torch.cuda.is_available())

In [ ]:
# 6. Direct Metric Training & Category-Binned Validation Loop
best_mae = 999.0
print(f"[Train] Starting {epochs} Epochs of Direct Metric Supervision with Blur Augmentation...")

for ep in range(1, epochs + 1):
    model.train()
    train_loss = 0.0
    t_start = time.time()
    
    for i, (rgb_b, agl_b) in enumerate(train_loader):
        rgb_b = rgb_b.to(device)
        agl_b = agl_b.to(device)
        
        # Apply blur augmentation
        rgb_b = apply_blur_augmentation(rgb_b, p=0.5)
        
        # Standard ImageNet normalization
        norm_rgb = (rgb_b - IMAGENET_MEAN) / IMAGENET_STD
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu', enabled=torch.cuda.is_available()):
            out = model(pixel_values=norm_rgb)
            p = out.predicted_depth
            if p.shape[-2:] != agl_b.shape[-2:]:
                p = F.interpolate(p.unsqueeze(1), size=agl_b.shape[-2:], mode='bilinear', align_corners=False).squeeze(1)
                
            # Direct metric output in metres
            pred_m = p.clamp(min=0.0)
            loss = criterion(pred_m, agl_b)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
        
        if (i + 1) % 50 == 0:
            print(f"Epoch [{ep:02d}/{epochs:02d}] Step [{i+1:04d}/{len(train_loader)}] Loss: {loss.item():.4f} LR: {scheduler.get_last_lr()[0]:.2e}")
            
    scheduler.step()
    
    # --- Category-Binned Validation Evaluation ---
    model.eval()
    cat_errors = {
        'ground': [],      # < 1.5m
        'canopy': [],      # 2 - 12m
        'buildings': [],   # 12 - 40m
        'skyscrapers': []  # > 40m
    }
    all_abs_errors = []
    all_sq_errors = []
    all_preds = []
    all_gts = []
    
    with torch.no_grad():
        for rgb_v, agl_v in val_loader:
            rgb_v = rgb_v.to(device)
            norm_rgb = (rgb_v - IMAGENET_MEAN) / IMAGENET_STD
            
            with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu', enabled=torch.cuda.is_available()):
                out = model(pixel_values=norm_rgb)
                p = out.predicted_depth
                if p.shape[-2:] != agl_v.shape[-2:]:
                    p = F.interpolate(p.unsqueeze(1), size=agl_v.shape[-2:], mode='bilinear', align_corners=False).squeeze(1)
                pred_m = p.clamp(min=0.0).cpu().numpy()
                gt_m = agl_v.numpy()
                
            for b in range(len(pred_m)):
                v = np.isfinite(gt_m[b]) & (gt_m[b] >= 0.0)
                if not np.any(v):
                    continue
                diff = np.abs(pred_m[b][v] - gt_m[b][v])
                sq_diff = (pred_m[b][v] - gt_m[b][v]) ** 2
                all_abs_errors.append(diff)
                all_sq_errors.append(sq_diff)
                all_preds.append(pred_m[b][v])
                all_gts.append(gt_m[b][v])
                
                # Category binning
                gt_b = gt_m[b][v]
                m_ground = gt_b < 1.5
                m_canopy = (gt_b >= 2.0) & (gt_b < 12.0)
                m_bldg = (gt_b >= 12.0) & (gt_b <= 40.0)
                m_tall = gt_b > 40.0
                
                if np.any(m_ground): cat_errors['ground'].append(diff[m_ground])
                if np.any(m_canopy): cat_errors['canopy'].append(diff[m_canopy])
                if np.any(m_bldg): cat_errors['buildings'].append(diff[m_bldg])
                if np.any(m_tall): cat_errors['skyscrapers'].append(diff[m_tall])
                
    flat_abs = np.concatenate(all_abs_errors) if all_abs_errors else np.array([0.0])
    flat_sq = np.concatenate(all_sq_errors) if all_sq_errors else np.array([0.0])
    val_mae = float(np.mean(flat_abs))
    val_rmse = float(np.sqrt(np.mean(flat_sq)))
    
    flat_p = np.concatenate(all_preds) if all_preds else np.array([0.0])
    flat_g = np.concatenate(all_gts) if all_gts else np.array([0.0])
    r_corr = float(pearsonr(flat_g[::10], flat_p[::10])[0]) if len(flat_p) > 10 else 0.0
    
    mae_ground = float(np.mean(np.concatenate(cat_errors['ground']))) if cat_errors['ground'] else 0.0
    mae_canopy = float(np.mean(np.concatenate(cat_errors['canopy']))) if cat_errors['canopy'] else 0.0
    mae_bldg = float(np.mean(np.concatenate(cat_errors['buildings']))) if cat_errors['buildings'] else 0.0
    mae_tall = float(np.mean(np.concatenate(cat_errors['skyscrapers']))) if cat_errors['skyscrapers'] else 0.0
    
    elapsed = time.time() - t_start
    avg_train_loss = train_loss / max(len(train_loader), 1)
    print(f"\n{'='*70}")
    print(f"==> Epoch [{ep:02d}/{epochs:02d}] Summary (Time: {elapsed:.1f}s):")
    print(f"    Train Loss: {avg_train_loss:.4f}")
    print(f"    OVERALL VAL MAE : {val_mae:.2f} m  (RMSE: {val_rmse:.2f}m | r: {r_corr:.3f})")
    print(f"    -- Ground (<1.5m)       : {mae_ground:.2f} m")
    print(f"    -- Canopy (2-12m)       : {mae_canopy:.2f} m")
    print(f"    -- Buildings (12-40m)   : {mae_bldg:.2f} m")
    print(f"    -- Skyscrapers (>40m)   : {mae_tall:.2f} m")
    print(f"{'='*70}\n")
    
    if val_mae < best_mae:
        best_mae = val_mae
        os.makedirs('output', exist_ok=True)
        save_path = 'output/depthwizard_best_gamus.pt'
        torch.save(model.state_dict(), save_path)
        meta = {
            'epoch': ep,
            'val_mae': round(val_mae, 3),
            'val_rmse': round(val_rmse, 3),
            'pearson_r': round(r_corr, 3),
            'categories': {
                'ground_mae': round(mae_ground, 3),
                'canopy_mae': round(mae_canopy, 3),
                'buildings_mae': round(mae_bldg, 3),
                'skyscrapers_mae': round(mae_tall, 3)
            }
        }
        with open('output/depthwizard_best_metrics.json', 'w') as f:
            json.dump(meta, f, indent=2)
        print(f"*** NEW BEST MODEL CHECKPOINT SAVED: MAE = {val_mae:.2f} m ***\n")

print(f"\nTraining Complete! Best model (MAE: {best_mae:.2f}m) saved to output/depthwizard_best_gamus.pt")